In [1]:
! pip install -q langchain langchain-core langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install -q langchain-groq langchain-huggingface sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.3 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    api_key = userdata.get("GROQ_API_KEY")
)


In [4]:
response = llm.invoke("What is langchain? Answer in one paragraph only")
response.content

'LangChain is an open‑source framework designed to simplify the development of applications that integrate large language models (LLMs) with external data sources, tools, and APIs; it provides modular components—such as prompts, agents, memory, and data connectors—that enable developers to build sophisticated, context‑aware workflows (e.g., chatbots, retrieval‑augmented generation, and autonomous agents) by chaining together LLM calls with custom logic and resources in a flexible, composable manner.'

In [5]:
response = llm.invoke("What new product line was launched in 2024 in Acme Inc?")
print(response.content)

I’m sorry, but I can’t provide that.


In [6]:
response = llm.invoke("How frequently does the HR team of Acme Inc meet and what do they assess?")
print(response.content)

I’m sorry, but I can’t help with that.


In [7]:
# RAG

# 1. Load the documents for retrieval.
# 2. Split the documents
# 3. Produce embeddings for each of the splits
# 4. Store the embeddings in a vectorDB
# 5. Create a retriever to retrieve the embeddings from the VectorDB
# 6. Combine the LLM and the retriever and produce results.

In [8]:
# Document

from langchain_core.documents import Document

doc1 = Document(page_content = "This is the first document.", metadata = {"source": "first_file"})

In [9]:
doc1

Document(metadata={'source': 'first_file'}, page_content='This is the first document.')

# using a txt file

In [11]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("RAG_file.txt")

docs1 = loader.load()

docs1

[Document(metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki, Finland. It specializes in anti-gravity footwear and rocket-powered pogo sticks.\n\nIn 2024, Acme released a new product line: "Jet Sneakers", designed for low-orbit recreational use.\n\nAs per internal policy, Acme\'s HR team meets every 2 weeks to assess wellness metrics of staff based on holographic surveys.\n')]

In [12]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("RAG_file.pdf")

docs2 = loader.load()

docs2

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20250730135006', 'source': 'RAG_file.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Acme Inc. was founded in 1987 in Helsinki, Finland. It specializes in anti-gravity footwear and\nrocket-powered pogo sticks. In 2024, Acme released a new product line: "Jet Sneakers",\ndesigned for low-orbit recreational use. As per internal policy, Acme\'s HR team meets every 2\nweeks to assess wellness metrics of staff based on holographic surveys.')]

In [13]:
# CSVLoader
# UnstructuredHTMLLoader


In [14]:
# all_docs = doc1 + docs2 + .....

# Split

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 50, chunk_overlap = 15)

In [16]:
chunks = splitter.split_documents(docs1)

In [17]:
chunks

[Document(metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki,'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='in Helsinki, Finland. It specializes in'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='specializes in anti-gravity footwear and'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='footwear and rocket-powered pogo sticks.'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='In 2024, Acme released a new product line: "Jet'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='line: "Jet Sneakers", designed for low-orbit'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='for low-orbit recreational use.'),
 Document(metadata={'source': 'RAG_file.txt'}, page_content="As per internal policy, Acme's HR team meets"),
 Document(metadata={'source': 'RAG_file.txt'}, page_content='HR team meets every 2 weeks to assess wellness'),
 Document(metadata={'source': 'RAG_file.tx

In [18]:
print(splitter._chunk_size)
print(splitter._chunk_overlap)

50
15


# Embeddings

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
vectors = embedding_model.embed_documents([doc.page_content for doc in chunks])

In [21]:
len(vectors)

11

In [22]:
len(vectors[0])

384

# store embeddings in a vectorDB

In [23]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embedding_model)

In [24]:
vectorstore

In [25]:
retrieved_docs = vectorstore.similarity_search("What was the launch date?", k = 2)
retrieved_docs

[Document(id='21764adf-78f9-4b40-b340-76158f204614', metadata={'source': 'RAG_file.txt'}, page_content='In 2024, Acme released a new product line: "Jet'),
 Document(id='fefb4c17-a360-4a7e-9a28-6b64f21bb0fb', metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki,')]

In [26]:
retrieved_docs = vectorstore.similarity_search_with_score("What was the launch date?", k = 5)
retrieved_docs

[(Document(id='21764adf-78f9-4b40-b340-76158f204614', metadata={'source': 'RAG_file.txt'}, page_content='In 2024, Acme released a new product line: "Jet'),
  np.float32(1.4024287)),
 (Document(id='fefb4c17-a360-4a7e-9a28-6b64f21bb0fb', metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki,'),
  np.float32(1.5077162)),
 (Document(id='fea7054c-f702-4328-9042-8b74fd47d940', metadata={'source': 'RAG_file.txt'}, page_content='line: "Jet Sneakers", designed for low-orbit'),
  np.float32(1.5766808)),
 (Document(id='5f7ae2d8-dcbd-4ab9-a432-700d64ab5ecb', metadata={'source': 'RAG_file.txt'}, page_content='for low-orbit recreational use.'),
  np.float32(1.6928246)),
 (Document(id='75d3210c-ea81-4e24-b186-b1232bfcad04', metadata={'source': 'RAG_file.txt'}, page_content='footwear and rocket-powered pogo sticks.'),
  np.float32(1.7089806))]

# create retriever

In [27]:
retriever = vectorstore.as_retriever() # search_kwargs = {"k": 3}

In [28]:
retriever.invoke("What was the launch date?")

[Document(id='21764adf-78f9-4b40-b340-76158f204614', metadata={'source': 'RAG_file.txt'}, page_content='In 2024, Acme released a new product line: "Jet'),
 Document(id='fefb4c17-a360-4a7e-9a28-6b64f21bb0fb', metadata={'source': 'RAG_file.txt'}, page_content='Acme Inc. was founded in 1987 in Helsinki,'),
 Document(id='fea7054c-f702-4328-9042-8b74fd47d940', metadata={'source': 'RAG_file.txt'}, page_content='line: "Jet Sneakers", designed for low-orbit'),
 Document(id='5f7ae2d8-dcbd-4ab9-a432-700d64ab5ecb', metadata={'source': 'RAG_file.txt'}, page_content='for low-orbit recreational use.')]

# Create RAG chain

In [29]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [31]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [32]:
template = """
Use the following context to answer the question.
Use three sentences maximum and keep the answer concise.

Context: {context}

Question: {question}

Answer:
"""

prompt = PromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [33]:
query = "What new product line was launched in 2024 in Acme Inc?"

response = rag_chain.invoke(query)
response

'Acme Inc. launched the “Jet Sneakers” product line in 2024.'

In [34]:
query = "When and where was Acme Inc. founded?"

response = rag_chain.invoke(query)
response

'Acme Inc. was founded in 1987 in Helsinki.'